# HomeGuard Security System Simulator 
## Author: Moritz Glas 
## Description: A smart home monitoring system that processes and triggers alerst for security, safety and comfort issues. 

In [104]:
import random  # used to simulate sensor readings
from datetime import datetime  # used for timestamping readings and alerts 

In [105]:
# System configuration: possible modes and alert levels
Home_modes = ["HOME", "AWAY", "SLEEP"]
Alert_Severities = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

# Current system state (nobody is home)
current_mode = "AWAY"


## Step 2

In [106]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """ Creates a dictionary that stores all information about one sensor."""

    sensor = {
        "id": sensor_id,
        "location": location, 
        "type": sensor_type,        # "motion", "temeperature", "door" or "smoke"
        "threshold": threshold,     # only needed for temperature sensors
        "current_value": None       # no reading yet 
    }
    return sensor 

def create_alert(severity, message, sensor_id, timestamp):
    """Creates a dictionary that stores all information about one alert."""
    alert = {
        "severity": severity,       # "LOW", "MEDIUM", "HIGH" or "CRITICAL"
        "message": message, 
        "sensor_id": sensor_id,     # which sensor triggered the alert
        "timestamp": timestamp
    }
    return alert

# Sensors installed in the Peterson vacation home 
sensors = [
    create_sensor("Motion_01", "Living Room", "motion"), 
    create_sensor("Temp_01", "Kitchen", "temperature", threshold=35),  
    create_sensor("Door_01", "Front Door", "door"), 
    create_sensor("Smoke_01", "Bedroom", "smoke")
]



In [107]:
# Checkpoint: verify that all sensors were created correctly 
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors: 
    print(f" - {sensor['id']}: {sensor['location']} ({sensor['type']})")



Initialized 4 sensors
 - Motion_01: Living Room (motion)
 - Temp_01: Kitchen (temperature)
 - Door_01: Front Door (door)
 - Smoke_01: Bedroom (smoke)


## Step 3 

In [108]:
def is_abnormal_reading(sensor, reading_value):
    """Returns True if the reading is abnormal for this type of sensor."""
    sensor_type = sensor ["type"]

    # Temperature: below 35 (frozen pipes) or above 95 (equipment failure)
    if sensor_type == "temperature":
        if reading_value < 35 or reading_value > 95: 
            return True 

    # Motion: any detected movement 
    elif sensor_type == "motion":
        if reading_value == True: 
            return True 

    # Door: door is open 
    elif sensor_type == "door":
        if reading_value == "OPEN": 
            return True 

    # Smoke: smoke detected 
    elif sensor_type == "smoke":
        if reading_value == "DETECTED": 
            return True 

    # No condition met -> reading is normal 
    return False
        
  



In [109]:
def should_trigger_security_alert(sensor, reading_value, system_mode):
    """Returns True if motion or an open door is detected while in AWAY mode."""
    # Security alerts only matter when nobody is home
    if system_mode != "AWAY": 
        return False
    if sensor ["type"] == "motion" and reading_value == True: 
        return True
    if sensor ["type"] == "door" and reading_value == "OPEN":
        return True

    return False


In [110]:
# Checkpoint: test temperature check and security alert logic 
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")   # Should be True 
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")   # Should be False

motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # SHould be True 
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False 


34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


## Step 4

In [111]:
def generate_reading(sensor):
    """Simulates a sensor by returning a random value that macthes its type"""
    sensor_type = sensor["type"]
    if sensor_type == "temperature":
        return random.randint(30, 100)    # degrees Fahrenheit 
    elif sensor_type == "motion":
        return random.choice([True, False])
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke":
        return random.choice(["DETECTED", "CLEAR"])

    return None   # unknown sensor type 


In [112]:
def process_reading(sensor, reading_value, system_mode):
    """Checks one reading against all rules and returns a list of alerts"""
    alerts = []   # collects all alerts triggered by this reading 
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_type = sensor["type"]
    location = sensor["location"]

    # Security alerts: motion or open door while nobody is home 
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        if sensor_type == "motion":
            message = f"SECURITY: Motion detected in {location} while in AWAY mode!"
        else:
            message = f"SECURITY: {location} opened while in AWAY mode!"
        alerts.append(create_alert("HIGH", message, sensor["id"], timestamp))

    # Safety alerts: extreme temperatures or smoke, checked in every mode 
    if sensor_type == "temperature":
        if reading_value < 35:
            message = f"SAFETY: {location} at {reading_value}°F - risk of frozen pipes!"
            alerts.append(create_alert("CRITICAL", message, sensor["id"], timestamp))
        elif reading_value > 95:
            message = f"SAFETY: {location} at {reading_value}°F - possible equipment failure!"
            alerts.append(create_alert("CRITICAL", message, sensor["id"], timestamp))
    
    if sensor_type == "smoke" and reading_value == "DETECTED":
        message = f"SAFETY: Smoke detected in {location} - fire rik!"
        alerts.append(create_alert("CRITICAL", message, sensor["id"], timestamp))
   
    # Comfort notifications: only in HOME mode, temperature outside 65-75 degrees Fahrenheit 
    # Extreme values are excluded because they already trigger a safety alert
    if system_mode == "HOME" and sensor_type == "temperature":
        if 35 <= reading_value <= 95 and (reading_value < 65 or reading_value > 75):
            message = f"COMFORT: {location} at {reading_value}°F - outside comfort range (65-75°F)!"
            alerts.append(create_alert("LOW", message, sensor["id"], timestamp))
    return alerts
            

In [113]:
def trigger_alert(alert):
    """Prints an alert with a symbol that matches its severity"""
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

def log_event(message, timestamp=None): 
    """Prints a log message; uses the current time if no timestamp is given"""
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}]: {message}")
    

In [114]:
# Checkpoint: generate a random reading and process a forced motion reading 
test_sensor = sensors[0]   # motion sensor 
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")

alerts = process_reading(test_sensor, True, "AWAY")
if alerts: 
    trigger_alert(alerts[0])
    

Generated reading for Living Room: True
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!


## Step 5

In [115]:
class Sensor: 
    """Represents a sensor in the HomeGuard system."""

    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        """Stores the sensor's properties in the object (replaces create_sensor)."""
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None   # no reading yet 

    def read(self):
        """Generates a new reading, stores it in the object and returns it."""
        self.current_value = generate_reading({"type": self.type})
        return self.current_value
    

    def isAbnormal(self):
        """Checks the stores reading using the logic from Step 3."""
        if self.current_value is None:    # noting to check without a reading
            return False 
        return is_abnormal_reading({"type": self.type}, self.current_value)
    def reset(self):
        """Clears the stored reading."""
        self.current_value = None
    def __str__(self):
        """Defines how the sensor is displayed when printed."""
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"

# Sensors of the Peterson home, now as objects instead of dictionaries
sensor_objects = [
    Sensor("Motion_01", "Living Room", "motion"), 
    Sensor("Temp_01", "Kitchen", "temperature", threshold=35),  
    Sensor("Door_01", "Front Door", "door"), 
    Sensor("Smoke_01", "Bedroom", "smoke")
    ]

    

In [116]:
# Checkpoint: create a sensor object, read it and check the result 
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"isAbnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")


Sensor reading: 71
isAbnormal: False
Sensor info: TEST_001 (Test Room): 71


## Step 6

In [117]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """Runs the HomeGuard simulation for a given number of minutes and mode."""
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")

    # Fresh sensor objects for each simulation run 
    sensors = [
        Sensor("Motion_01", "Living Room", "motion"),
        Sensor("Temp_01", "Kitchen", "temperature", threshold=35),
        Sensor("Door_01", "Front Door", "door"),
        Sensor("Smoke_01", "Bedroom", "smoke")
    ]

    # Each iteration represents one minute 
    for minute in range(duration_minutes): 
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        # Read and evaluate every sensor 
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading in a format that fits the sensor type 
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")

            # Convert to dict because process_reading expects a dictionary 
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Show all alerts; notify the homeowner for serious ones
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

        import time
       
        time.sleep(0.5)   # short pause to keep the output readable
   
    print("\n" + "=" * 50)
    print("\nSimulation complete.")
    print("=" * 50)


# Start the simulation when the file is run directly 
if __name__ == "__main__":
    run_simulation(duration_minutes=3, system_mode="AWAY")
print("Simulation complete!") 

                

=== HomeGuard Security System ===
Mode: AWAY


Time: 19:20:02
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [19:20:02]: Sending notification to homeowner...
[READING] Kitchen Temperature: 38°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom - fire rik!
[LOG] [19:20:02]: Sending notification to homeowner...

Time: 19:20:03
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [19:20:03]: Sending notification to homeowner...
[READING] Kitchen Temperature: 61°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 19:20:03
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [19:20:03]: Sending notification to homeowner...
[READING] Kitchen Temperature: 31°F (Abnormal)
[ALE

## Step 7

In [118]:
# Security test: motion and open door should trigger HIGH alerts 
run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 19:20:04
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 64°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 19:20:04
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [19:20:04]: Sending notification to homeowner...
[READING] Kitchen Temperature: 58°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [19:20:04]: Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom - fire rik!
[LOG] [19:20:04]: Sending notification to homeowner...

Time: 19:20:05
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 85°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [19:20:05]: Sending notification to homeowner...

In [119]:
# Comfort test: no security alerts, but LOW comfort notifications outside 65-75 degrees Fahrenheit
run_simulation(duration_minutes=3, system_mode="HOME")

=== HomeGuard Security System ===
Mode: HOME


Time: 19:20:05
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 37°F (Abnormal)
[ALERT!] ℹ️ LOW: COMFORT: Kitchen at 37°F - outside comfort range (65-75°F)!
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom - fire rik!
[LOG] [19:20:05]: Sending notification to homeowner...

Time: 19:20:06
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 46°F (Abnormal)
[ALERT!] ℹ️ LOW: COMFORT: Kitchen at 46°F - outside comfort range (65-75°F)!
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom - fire rik!
[LOG] [19:20:06]: Sending notification to homeowner...

Time: 19:20:06
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 82°F (Abnormal)
[ALERT!] ℹ️ LOW: COMFORT: Kitchen at 82°F - outside comfort range (65-75°F)!
[READING] Front Door: CLOSED
[READING] Bedroom S

In [ ]:
# Sleep test: only safety alerts (extreme temperature or smoke) should appearb
run_simulation(duration_minutes=3, system_mode="SLEEP")

=== HomeGuard Security System ===
Mode: SLEEP


Time: 19:20:07
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 41°F (Abnormal)
[READING] Front Door: OPEN
[READING] Bedroom Smoke: CLEAR

Time: 19:20:08
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 49°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom - fire rik!
[LOG] [19:20:08]: Sending notification to homeowner...

Time: 19:20:08
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 45°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom - fire rik!
[LOG] [19:20:08]: Sending notification to homeowner...


Simulation complete.


## Code explanation 
My code creates four sensors (motion, temperature, door and smoke) that generate random readings every stimulated minute. Each reading is passed to process_reading, which uses if/else logic to check it against three rules. Security alerts for motion or an open door in away mode, safety alerts for extreme temperature or smoke in every mode, and comfort notifications for temperatures outside 65-75 degrees fahrenheit in home mode. Every triggered alert is stored as a dictionary, displayed with its severity, and serious alerts additionally log notification to the homeowner. 